# 🏥 CNN-Based ICD-10 Auto-Coding System
## Notebook 3: CNN Model Training

---

**⚠️ IMPORTANT: Use GPU Runtime!**
- Runtime → Change runtime type → T4 GPU

**This Notebook Covers**:
1. Load preprocessed data from Notebook 2
2. Build CNN architecture for multi-label classification
3. Train the model with early stopping
4. Evaluate on validation and test sets
5. Save trained model to Google Drive

**Input**: Preprocessed data from `train_test_split/`
**Output**: Trained CNN model, training history, metrics

**Estimated Runtime**: ~30-45 minutes (T4 GPU)

---

## 📋 Section 1: Setup & Load Data (~2 min)

In [ ]:
# Check GPU availability
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✓ GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU! Training will be SLOW.")
    print("  Go to: Runtime → Change runtime type → T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted!")

In [ ]:
# Imports
import numpy as np
import pandas as pd
import pickle
import json
import os
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Metrics
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report
)

print("✓ All imports successful!")

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive"
PROJECT_FOLDER = f"{DRIVE_BASE}/ICD10_Project"
DATA_FOLDER = f"{PROJECT_FOLDER}/data"
TRAIN_TEST_FOLDER = f"{DATA_FOLDER}/train_test_split"
MODELS_FOLDER = f"{PROJECT_FOLDER}/models"

# Input files
PREPROCESSED_DATA_FILE = f"{TRAIN_TEST_FOLDER}/preprocessed_data.pkl"
VOCAB_FILE = f"{TRAIN_TEST_FOLDER}/vocabulary.pkl"
LABEL_ENCODER_FILE = f"{TRAIN_TEST_FOLDER}/label_encoder.pkl"

# Model hyperparameters
EMBEDDING_DIM = 128       # Word embedding size
NUM_FILTERS = 128         # CNN filters per kernel size
KERNEL_SIZES = [2, 3, 4, 5]  # Different n-gram sizes
DROPOUT_RATE = 0.5        # Dropout for regularization
HIDDEN_DIM = 256          # Dense layer size

# Training settings
BATCH_SIZE = 32           # Batch size
LEARNING_RATE = 0.001     # Initial learning rate
NUM_EPOCHS = 30           # Max epochs
EARLY_STOP_PATIENCE = 5   # Stop if no improvement
THRESHOLD = 0.5           # Classification threshold

# Create output directory
os.makedirs(MODELS_FOLDER, exist_ok=True)

print("Configuration set!")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Filters: {NUM_FILTERS} x {len(KERNEL_SIZES)} kernels")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")

In [ ]:
# Load preprocessed data
print("Loading preprocessed data...")

with open(PREPROCESSED_DATA_FILE, 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_val = data['X_val']
X_test = data['X_test']
Y_train = data['Y_train']
Y_val = data['Y_val']
Y_test = data['Y_test']

VOCAB_SIZE = data['vocab_size']
N_CLASSES = data['n_classes']
MAX_SEQ_LENGTH = data['max_sequence_length']

print(f"\n✓ Data loaded!")
print(f"  Train: {X_train.shape}")
print(f"  Val: {X_val.shape}")
print(f"  Test: {X_test.shape}")
print(f"\n  Vocab size: {VOCAB_SIZE}")
print(f"  Classes: {N_CLASSES}")
print(f"  Sequence length: {MAX_SEQ_LENGTH}")

In [ ]:
# Load label encoder for later analysis
with open(LABEL_ENCODER_FILE, 'rb') as f:
    label_encoder = pickle.load(f)

idx2code = label_encoder['idx2code']
code2idx = label_encoder['code2idx']
print(f"✓ Label encoder loaded with {len(idx2code)} codes")

---

## 🔧 Section 2: Create Data Loaders (~1 min)

In [ ]:
class ICD10Dataset(Dataset):
    """PyTorch Dataset for ICD-10 classification"""
    
    def __init__(self, X, Y):
        self.X = torch.LongTensor(X)
        self.Y = torch.FloatTensor(Y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


# Create datasets
train_dataset = ICD10Dataset(X_train, Y_train)
val_dataset = ICD10Dataset(X_val, Y_val)
test_dataset = ICD10Dataset(X_test, Y_test)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"✓ Data loaders created!")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

---

## 🧠 Section 3: Build CNN Model (~1 min)

In [ ]:
class TextCNN(nn.Module):
    """
    CNN for Multi-Label Text Classification.
    
    Architecture:
    - Embedding layer
    - Multiple parallel Conv1D layers (different kernel sizes)
    - Max pooling
    - Concatenation
    - Dense layers with dropout
    - Sigmoid output for multi-label
    """
    
    def __init__(self, vocab_size, embedding_dim, n_classes,
                 num_filters, kernel_sizes, dropout_rate, hidden_dim):
        super(TextCNN, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        
        # Convolutional layers with different kernel sizes
        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])
        
        # Fully connected layers
        total_filters = num_filters * len(kernel_sizes)
        self.fc1 = nn.Linear(total_filters, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, n_classes)
        
        # Dropout and batch normalization
        self.dropout = nn.Dropout(dropout_rate)
        self.bn = nn.BatchNorm1d(hidden_dim)
        
    def forward(self, x):
        # Embedding: (batch, seq_len) -> (batch, seq_len, embed_dim)
        x = self.embedding(x)
        
        # Transpose for Conv1d: (batch, embed_dim, seq_len)
        x = x.permute(0, 2, 1)
        
        # Apply convolutions with different kernel sizes
        conv_outputs = []
        for conv in self.convs:
            # Conv + ReLU + MaxPool
            c = F.relu(conv(x))
            c = F.max_pool1d(c, c.size(2)).squeeze(2)
            conv_outputs.append(c)
        
        # Concatenate all conv outputs
        x = torch.cat(conv_outputs, dim=1)
        
        # Fully connected layers
        x = self.dropout(x)
        x = F.relu(self.bn(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        
        # Sigmoid for multi-label
        return torch.sigmoid(x)


# Create model
model = TextCNN(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    n_classes=N_CLASSES,
    num_filters=NUM_FILTERS,
    kernel_sizes=KERNEL_SIZES,
    dropout_rate=DROPOUT_RATE,
    hidden_dim=HIDDEN_DIM
).to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "="*50)
print("CNN MODEL ARCHITECTURE")
print("="*50)
print(model)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross-Entropy for multi-label
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

print("✓ Loss function: Binary Cross-Entropy (BCE)")
print(f"✓ Optimizer: Adam (lr={LEARNING_RATE})")
print("✓ LR Scheduler: ReduceLROnPlateau")

---

## 🏋️ Section 4: Training Loop (~30-40 min)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    for X_batch, Y_batch in loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, Y_batch)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device, threshold=0.5):
    """Evaluate model on validation/test set"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)
            
            outputs = model(X_batch)
            loss = criterion(outputs, Y_batch)
            total_loss += loss.item()
            
            # Convert to predictions
            preds = (outputs >= threshold).float()
            
            all_probs.extend(outputs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(Y_batch.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Calculate metrics
    metrics = {
        'loss': total_loss / len(loader),
        'micro_f1': f1_score(all_labels, all_preds, average='micro', zero_division=0),
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'micro_precision': precision_score(all_labels, all_preds, average='micro', zero_division=0),
        'micro_recall': recall_score(all_labels, all_preds, average='micro', zero_division=0),
    }
    
    # Try to calculate AUC (may fail if some classes have no samples)
    try:
        metrics['roc_auc'] = roc_auc_score(all_labels, all_probs, average='micro')
    except:
        metrics['roc_auc'] = 0.0
    
    return metrics, all_preds, all_labels, all_probs


print("✓ Training functions defined!")

In [ ]:
# Training loop with early stopping
print("\n" + "="*60)
print("🏋️ STARTING TRAINING")
print("="*60)
print(f"Epochs: {NUM_EPOCHS} (max)")
print(f"Early stopping patience: {EARLY_STOP_PATIENCE}")
print(f"Device: {device}")
print("="*60 + "\n")

# Track history
history = {
    'train_loss': [],
    'val_loss': [],
    'val_micro_f1': [],
    'val_macro_f1': [],
    'val_roc_auc': []
}

best_val_loss = float('inf')
best_val_f1 = 0
patience_counter = 0
best_model_state = None

for epoch in range(NUM_EPOCHS):
    start_time = datetime.now()
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Evaluate on validation set
    val_metrics, _, _, _ = evaluate(model, val_loader, criterion, device, THRESHOLD)
    
    # Update learning rate
    scheduler.step(val_metrics['loss'])
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_metrics['loss'])
    history['val_micro_f1'].append(val_metrics['micro_f1'])
    history['val_macro_f1'].append(val_metrics['macro_f1'])
    history['val_roc_auc'].append(val_metrics['roc_auc'])
    
    # Calculate epoch time
    epoch_time = (datetime.now() - start_time).total_seconds()
    
    # Print progress
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_metrics['loss']:.4f} | "
          f"Val F1 (micro): {val_metrics['micro_f1']:.4f} | "
          f"Val AUC: {val_metrics['roc_auc']:.4f} | "
          f"Time: {epoch_time:.1f}s")
    
    # Check for improvement
    if val_metrics['micro_f1'] > best_val_f1:
        best_val_f1 = val_metrics['micro_f1']
        best_val_loss = val_metrics['loss']
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        print(f"         ↑ New best model! (F1: {best_val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n⚡ Early stopping at epoch {epoch+1}!")
            break

# Restore best model
if best_model_state:
    model.load_state_dict(best_model_state)

print("\n" + "="*60)
print("✓ TRAINING COMPLETE")
print(f"  Best Val F1 (micro): {best_val_f1:.4f}")
print(f"  Best Val Loss: {best_val_loss:.4f}")
print("="*60)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training vs Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 Score
axes[1].plot(history['val_micro_f1'], label='Micro F1', linewidth=2)
axes[1].plot(history['val_macro_f1'], label='Macro F1', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('Validation F1 Scores')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# ROC AUC
axes[2].plot(history['val_roc_auc'], label='ROC AUC', linewidth=2, color='green')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('ROC AUC')
axes[2].set_title('Validation ROC AUC')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{MODELS_FOLDER}/training_history.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Training history plot saved!")

---

## 📊 Section 5: Evaluation (~3 min)

In [ ]:
# Evaluate on test set
print("\n" + "="*60)
print("📊 EVALUATING ON TEST SET")
print("="*60)

test_metrics, test_preds, test_labels, test_probs = evaluate(
    model, test_loader, criterion, device, THRESHOLD
)

print(f"\nTest Results:")
print(f"  Loss: {test_metrics['loss']:.4f}")
print(f"  Micro F1: {test_metrics['micro_f1']:.4f}")
print(f"  Macro F1: {test_metrics['macro_f1']:.4f}")
print(f"  Micro Precision: {test_metrics['micro_precision']:.4f}")
print(f"  Micro Recall: {test_metrics['micro_recall']:.4f}")
print(f"  ROC AUC: {test_metrics['roc_auc']:.4f}")

In [ ]:
# Per-class F1 scores (top 20)
print("\n" + "="*60)
print("TOP 20 CODES BY F1 SCORE")
print("="*60)

# Calculate per-class F1
per_class_f1 = []
for i in range(N_CLASSES):
    y_true = test_labels[:, i]
    y_pred = test_preds[:, i]
    support = y_true.sum()
    
    if support > 0:
        f1 = f1_score(y_true, y_pred, zero_division=0)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        per_class_f1.append({
            'code': idx2code[i],
            'f1': f1,
            'precision': precision,
            'recall': recall,
            'support': int(support)
        })

# Sort by F1
per_class_f1 = sorted(per_class_f1, key=lambda x: x['f1'], reverse=True)

print(f"{'Code':<10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'Support':>8}")
print("-" * 50)
for item in per_class_f1[:20]:
    print(f"{item['code']:<10} {item['f1']:>8.3f} {item['precision']:>10.3f} "
          f"{item['recall']:>8.3f} {item['support']:>8d}")

In [ ]:
# Confusion analysis
print("\n" + "="*60)
print("PREDICTION ANALYSIS")
print("="*60)

# Predictions per document
preds_per_doc = test_preds.sum(axis=1)
labels_per_doc = test_labels.sum(axis=1)

print(f"\nPredictions per document:")
print(f"  Predicted avg: {preds_per_doc.mean():.1f}")
print(f"  Actual avg: {labels_per_doc.mean():.1f}")
print(f"  Predicted min/max: {preds_per_doc.min():.0f} / {preds_per_doc.max():.0f}")
print(f"  Actual min/max: {labels_per_doc.min():.0f} / {labels_per_doc.max():.0f}")

# Exact match ratio
exact_matches = np.all(test_preds == test_labels, axis=1).sum()
print(f"\nExact match ratio: {exact_matches}/{len(test_preds)} ({exact_matches/len(test_preds)*100:.1f}%)")

# Partial match analysis
correct_any = (test_preds * test_labels).sum(axis=1) > 0
print(f"Documents with at least 1 correct prediction: {correct_any.sum()}/{len(test_preds)} ({correct_any.mean()*100:.1f}%)")

In [ ]:
# Visualize F1 distribution
f1_values = [item['f1'] for item in per_class_f1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# F1 distribution
axes[0].hist(f1_values, bins=20, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(f1_values):.3f}')
axes[0].set_xlabel('F1 Score')
axes[0].set_ylabel('Number of Codes')
axes[0].set_title('Per-Class F1 Distribution')
axes[0].legend()

# Top 10 codes
top10 = per_class_f1[:10]
codes = [item['code'] for item in top10]
f1s = [item['f1'] for item in top10]
axes[1].barh(codes[::-1], f1s[::-1])
axes[1].set_xlabel('F1 Score')
axes[1].set_title('Top 10 Codes by F1 Score')

plt.tight_layout()
plt.savefig(f"{MODELS_FOLDER}/evaluation_metrics.png", dpi=150, bbox_inches='tight')
plt.show()

---

## 💾 Section 6: Save Model (~1 min)

In [ ]:
# Save model
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_path = f"{MODELS_FOLDER}/icd10_cnn_{timestamp}.pt"

# Save complete model info
model_info = {
    'model_state_dict': model.state_dict(),
    'model_config': {
        'vocab_size': VOCAB_SIZE,
        'embedding_dim': EMBEDDING_DIM,
        'n_classes': N_CLASSES,
        'num_filters': NUM_FILTERS,
        'kernel_sizes': KERNEL_SIZES,
        'dropout_rate': DROPOUT_RATE,
        'hidden_dim': HIDDEN_DIM
    },
    'training_config': {
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'epochs_trained': len(history['train_loss']),
        'threshold': THRESHOLD
    },
    'metrics': {
        'best_val_f1': best_val_f1,
        'best_val_loss': best_val_loss,
        'test_metrics': test_metrics
    },
    'history': history,
    'timestamp': timestamp
}

torch.save(model_info, model_path)
print(f"✓ Model saved to: {model_path}")

# Also save as 'latest' for easy loading
latest_path = f"{MODELS_FOLDER}/icd10_cnn_latest.pt"
torch.save(model_info, latest_path)
print(f"✓ Also saved as: {latest_path}")

In [ ]:
# Save per-class metrics
metrics_df = pd.DataFrame(per_class_f1)
metrics_path = f"{MODELS_FOLDER}/per_class_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)
print(f"✓ Per-class metrics saved to: {metrics_path}")

# Save training summary
summary = {
    'training_date': timestamp,
    'total_documents': len(X_train) + len(X_val) + len(X_test),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'vocab_size': VOCAB_SIZE,
    'n_classes': N_CLASSES,
    'epochs_trained': len(history['train_loss']),
    'best_val_f1': float(best_val_f1),
    'test_micro_f1': float(test_metrics['micro_f1']),
    'test_macro_f1': float(test_metrics['macro_f1']),
    'test_roc_auc': float(test_metrics['roc_auc']),
    'model_path': model_path
}

summary_path = f"{MODELS_FOLDER}/training_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✓ Training summary saved to: {summary_path}")

---

## ✅ Section 7: Final Summary

In [ ]:
print("\n" + "="*60)
print("🎉 PHASE 3 COMPLETE: CNN TRAINING")
print("="*60)

print("\n📊 Model Performance:")
print(f"  Test Micro F1: {test_metrics['micro_f1']:.4f}")
print(f"  Test Macro F1: {test_metrics['macro_f1']:.4f}")
print(f"  Test ROC AUC: {test_metrics['roc_auc']:.4f}")
print(f"  Precision: {test_metrics['micro_precision']:.4f}")
print(f"  Recall: {test_metrics['micro_recall']:.4f}")

print("\n📁 Saved Files:")
print(f"  {model_path}")
print(f"  {metrics_path}")
print(f"  {summary_path}")
print(f"  {MODELS_FOLDER}/training_history.png")
print(f"  {MODELS_FOLDER}/evaluation_metrics.png")

print("\n🏆 Top 5 Best Predicted Codes:")
for i, item in enumerate(per_class_f1[:5], 1):
    print(f"  {i}. {item['code']}: F1={item['f1']:.3f}")

print("\n📌 Next Steps:")
print("  1. Run Notebook 4: Inference & Deployment")
print("  2. Use model for predictions on new documents")
print("  3. Fine-tune hyperparameters if needed")

print("\n" + "="*60)